In [1]:
import re
import pandas as pd
import numpy as np

In [38]:
INPUT_FILE = "C:/Users/Asha Anandan/Documents/Study Materials/Portfolio Projects/Project 1 E-commerce Customer Intelligence/online_retail_II.xlsx"
OUT_DIR = "C:/Users/Asha Anandan/Documents/Study Materials/Portfolio Projects/Project 1 E-commerce Customer Intelligence/Outputs"

In [3]:
# Stock codes that are NOT physical products (services, fees, adjustments...)
LINE_CATEGORY_BY_CODE = {
    "POST": "Postage/Carriage", "DOT": "Postage/Carriage",
    "C2": "Postage/Carriage", "C3": "Postage/Carriage",
    "D": "Discount",
    "M": "Manual",
    "S": "Sample",
    "BANK CHARGES": "Fee/Adjustment", "AMAZONFEE": "Fee/Adjustment",
    "ADJUST": "Fee/Adjustment", "ADJUST2": "Fee/Adjustment",
    "CRUK": "Fee/Adjustment", "B": "Fee/Adjustment",
    "TEST001": "Test", "TEST002": "Test",
}

COUNTRY_MAP = {
    "EIRE": "Ireland", "RSA": "South Africa", "USA": "United States",
    "Korea": "South Korea", "Unspecified": "Unknown",
    "European Community": "Unknown",
}

In [4]:
issue_log = []          # list of dicts, turned into a DataFrame at the end
excluded_parts = []     # list of DataFrames, each tagged with ExclusionReason

In [5]:
def log(step, issue, rows, action):
    issue_log.append({"Step": step, "Issue": issue,
                      "RowsAffected": int(rows), "Action": action})
    print(f"[{step}] {issue}: {int(rows):,} rows -> {action}")

In [6]:
def exclude(df, mask, reason):
    """Move rows matching `mask` into the audit table and return the remainder."""
    out = df[mask].copy()
    out["ExclusionReason"] = reason
    excluded_parts.append(out)
    return df[~mask].copy()

In [8]:
# STEP 1 - LOAD & COMBINE BOTH TABS
# ----------------------------------------------------------------------------
# IDs are read as text: StockCode contains letters and Invoice can start with
# 'C'/'A'; reading as numbers would corrupt them (e.g. 85123A -> NaN).
xl = pd.ExcelFile(INPUT_FILE)

In [9]:
frames = []
for sheet in xl.sheet_names:
    part = xl.parse(sheet, dtype={"Invoice": str, "StockCode": str,
                                  "Description": str, "Country": str})
    part["SourceSheet"] = sheet
    part["SourceRow"] = np.arange(2, len(part) + 2)   # Excel row number
    frames.append(part)
    print(f"Loaded '{sheet}': {len(part):,} rows")

Loaded 'Year 2009-2010': 525,461 rows
Loaded 'Year 2010-2011': 541,910 rows


In [10]:
df = pd.concat(frames, ignore_index=True)
ROWS_RAW = len(df)
log("1 Load", "Rows loaded from both tabs", ROWS_RAW, "Concatenated")

[1 Load] Rows loaded from both tabs: 1,067,371 rows -> Concatenated


In [12]:
# STEP 2 - STANDARDISE COLUMN NAMES (match the data dictionary)

df = df.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice",
                        "Customer ID": "CustomerID"})
log("2 Schema", "Column names differ from data dictionary "
    "(Invoice/Price/Customer ID)", 0,
    "Renamed to InvoiceNo/UnitPrice/CustomerID")

[2 Schema] Column names differ from data dictionary (Invoice/Price/Customer ID): 0 rows -> Renamed to InvoiceNo/UnitPrice/CustomerID


In [13]:
df.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceSheet,SourceRow
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,2
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,3


In [14]:
# STEP 3 - TEXT HYGIENE (whitespace, case)
# ----------------------------------------------------------------------------
for col in ["InvoiceNo", "StockCode", "Description", "Country"]:
    before = df[col].copy()
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    log("3 Text", f"{col}: leading/trailing/double whitespace",
        (before.fillna("") != df[col].fillna("")).sum(),
        "Trimmed and collapsed to single spaces")

[3 Text] InvoiceNo: leading/trailing/double whitespace: 0 rows -> Trimmed and collapsed to single spaces
[3 Text] StockCode: leading/trailing/double whitespace: 1 rows -> Trimmed and collapsed to single spaces
[3 Text] Description: leading/trailing/double whitespace: 247,906 rows -> Trimmed and collapsed to single spaces
[3 Text] Country: leading/trailing/double whitespace: 0 rows -> Trimmed and collapsed to single spaces


In [15]:
# Stock codes: 183 codes appear in both cases (15056BL vs 15056bl, m vs M)
before = df["StockCode"].copy()
df["StockCode"] = df["StockCode"].str.upper()
log("3 Text", "StockCode mixed case (e.g. 15056bl / 15056BL)",
    (before != df["StockCode"]).sum(), "Upper-cased")

[3 Text] StockCode mixed case (e.g. 15056bl / 15056BL): 3,471 rows -> Upper-cased


In [16]:
# Description: store an UPPER-case copy; the raw text is kept for audit.
df["DescriptionRaw"] = df["Description"]
df["Description"] = df["Description"].str.upper()

In [17]:
# STEP 4 - DATA TYPES
# ----------------------------------------------------------------------------
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# CustomerID arrived as float because of NaNs (13085.0). Use nullable integer.
non_int = (df["CustomerID"].dropna() % 1 != 0).sum()
df["CustomerID"] = df["CustomerID"].astype("Int64")
log("4 Types", "CustomerID stored as float (because of missing values)",
    df["CustomerID"].notna().sum(),
    f"Converted to nullable Int64 ({non_int} non-integer values found)")
log("4 Types", "Unparseable dates / numerics",
    df[["InvoiceDate", "Quantity", "UnitPrice"]].isna().any(axis=1).sum(),
    "None found" if not df[["InvoiceDate", "Quantity", "UnitPrice"]]
    .isna().any(axis=1).any() else "Review")

[4 Types] CustomerID stored as float (because of missing values): 824,364 rows -> Converted to nullable Int64 (0 non-integer values found)
[4 Types] Unparseable dates / numerics: 0 rows -> None found


In [18]:
# Dictionary says: 6-digit number, prefix 'C' = cancellation.
# Reality: there is also an 'A' prefix (6 rows) = accounting "Adjust bad debt".
df["InvoicePrefix"] = df["InvoiceNo"].str.extract(r"^([A-Za-z])", expand=False).str.upper()
df["InvoiceType"] = np.select(
    [df["InvoicePrefix"] == "C", df["InvoicePrefix"] == "A"],
    ["Cancellation", "Adjustment"], default="Sale")
valid_format = df["InvoiceNo"].str.fullmatch(r"[CA]?\d{6}")
log("5 Invoice", "InvoiceNo not matching [C|A]+6 digits", (~valid_format).sum(),
    "None found" if valid_format.all() else "Review")
log("5 Invoice", "Cancellation invoices ('C' prefix)",
    (df["InvoiceType"] == "Cancellation").sum(), "Flagged InvoiceType=Cancellation")
log("5 Invoice", "Undocumented 'A' prefix invoices (bad-debt adjustments)",
    (df["InvoiceType"] == "Adjustment").sum(),
    "Flagged InvoiceType=Adjustment (excluded in step 9)")


[5 Invoice] InvoiceNo not matching [C|A]+6 digits: 0 rows -> None found
[5 Invoice] Cancellation invoices ('C' prefix): 19,494 rows -> Flagged InvoiceType=Cancellation
[5 Invoice] Undocumented 'A' prefix invoices (bad-debt adjustments): 6 rows -> Flagged InvoiceType=Adjustment (excluded in step 9)


In [19]:
# OVERLAP BETWEEN THE TWO TABS
# ----------------------------------------------------------------------------
# Tab 1 runs 2009-12-01..2010-12-09, tab 2 runs 2010-12-01..2011-12-09, so
# 1-9 Dec 2010 is present in BOTH tabs.
s1 = df[df["SourceSheet"] == xl.sheet_names[0]]
s2 = df[df["SourceSheet"] == xl.sheet_names[1]]
shared_invoices = set(s1["InvoiceNo"]) & set(s2["InvoiceNo"])

In [20]:
shared_invoices

{'536876',
 '538168',
 'C536379',
 '538008',
 '538154',
 '537811',
 '536988',
 '536526',
 'C536383',
 '537019',
 '536800',
 '537509',
 '537391',
 '538036',
 '537883',
 '536806',
 '537021',
 '537213',
 '537616',
 '537683',
 '537036',
 '537664',
 '536857',
 '536635',
 '538092',
 '537743',
 '537212',
 '537222',
 '538159',
 '538047',
 '537601',
 '537004',
 'C538082',
 '537010',
 '538165',
 '537111',
 '537871',
 '537054',
 '537142',
 '537605',
 '537201',
 '536781',
 '536371',
 'C537821',
 'C538083',
 '536639',
 '536920',
 '538076',
 '537879',
 '537014',
 '536614',
 '536673',
 '537438',
 '536589',
 '537981',
 '536782',
 '536551',
 '536604',
 'C536816',
 '537381',
 '536840',
 '536747',
 '536591',
 '537037',
 'C537157',
 '536740',
 '537211',
 '537409',
 'C537684',
 '537758',
 '538046',
 '536615',
 '537458',
 '537015',
 'C538118',
 '537435',
 'C537756',
 '538116',
 '536365',
 '537035',
 '537208',
 '536803',
 '537823',
 '536790',
 'C538062',
 '537390',
 '537766',
 '536941',
 '537806',
 '537051',

In [21]:
# Prove the copies are identical before we drop one of them.
key = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate",
       "UnitPrice", "CustomerID", "Country"]
def row_hashes(frame):
    """One hash per row (NaN-safe) so the two copies can be compared as multisets."""
    return pd.util.hash_pandas_object(frame[key].astype(object).where(frame[key].notna(), "<NA>")
                                      .astype(str), index=False).value_counts().sort_index()

In [22]:
a = row_hashes(s1[s1["InvoiceNo"].isin(shared_invoices)])
b = row_hashes(s2[s2["InvoiceNo"].isin(shared_invoices)])
identical = a.sort_index().equals(b.sort_index())
print(f"Overlap copies identical: {identical}")
assert identical, "Overlapping invoices differ between tabs - investigate!"

mask = (df["SourceSheet"] == xl.sheet_names[1]) & df["InvoiceNo"].isin(shared_invoices)
df = exclude(df, mask, "Cross-tab duplicate (1-9 Dec 2010 present in both tabs)")
log("6 Overlap", f"{len(shared_invoices):,} invoices present in both tabs "
    "(1-9 Dec 2010)", mask.sum(), "Kept the Tab-1 copy, dropped the Tab-2 copy")

Overlap copies identical: True
[6 Overlap] 1,088 invoices present in both tabs (1-9 Dec 2010): 22,523 rows -> Kept the Tab-1 copy, dropped the Tab-2 copy


In [24]:
#  DUPLICATE LINES WITHIN A TAB
# ----------------------------------------------------------------------------
# Duplicate definition: same invoice + item + quantity + price + customer + country. 
# Description and timestamp are deliberately NOT part of the key,
# because both are tidied later (typo variants, 1-9 minute timestamp drift inside an invoice) and would otherwise hide duplicates.
DROP_EXACT_DUPLICATES = True

dup_key = ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "Country"]
dup_mask = df.duplicated(subset=dup_key, keep="first")
if DROP_EXACT_DUPLICATES:
    df = exclude(df, dup_mask, "Duplicate line (same invoice, item, quantity, price, customer, country)")
    log("7 Duplicates", "Duplicate lines within a tab (same invoice/item/qty/price)", dup_mask.sum(),
        "Kept first occurrence, removed the rest")
else:
    df["IsDuplicate"] = dup_mask
    log("7 Duplicates", "Duplicate lines within a tab (same invoice/item/qty/price)", dup_mask.sum(),
        "Flagged only (IsDuplicate)")


[7 Duplicates] Duplicate lines within a tab (same invoice/item/qty/price): 11,816 rows -> Kept first occurrence, removed the rest


In [25]:
# LINE CLASSIFICATION (product vs service / fee / test)
# ----------------------------------------------------------------------------
# Dictionary says StockCode = 5-digit number. Reality: letter suffixes
# (85123A), plus codes like POST, DOT, M, D, BANK CHARGES, gift_0001_20...
std_code = df["StockCode"].str.fullmatch(r"\d{5}[A-Z]{0,2}")
df["LineCategory"] = df["StockCode"].map(LINE_CATEGORY_BY_CODE)
df.loc[df["StockCode"].str.startswith("GIFT_"), "LineCategory"] = "Gift Voucher"
df["LineCategory"] = df["LineCategory"].fillna("Product")
df["IsStandardStockCode"] = std_code
df["IsProduct"] = df["LineCategory"] == "Product"

log("8 Classify", "Non-standard StockCodes (not 5 digits + optional letters)",
    (~std_code).sum(),
    "Kept; classified in LineCategory (Postage, Discount, Manual, Fee, "
    "Voucher, Test) or left as Product (e.g. DCGS gift sets)")
log("8 Classify", "Non-product lines (everything except LineCategory=Product)",
    (~df["IsProduct"]).sum(), "Flagged with IsProduct / LineCategory")

[8 Classify] Non-standard StockCodes (not 5 digits + optional letters): 5,980 rows -> Kept; classified in LineCategory (Postage, Discount, Manual, Fee, Voucher, Test) or left as Product (e.g. DCGS gift sets)
[8 Classify] Non-product lines (everything except LineCategory=Product): 5,800 rows -> Flagged with IsProduct / LineCategory


In [26]:
# INVALID / NON-SALE ROWS  (-> excluded_rows)
# ----------------------------------------------------------------------------
# a. Test products
m = df["LineCategory"] == "Test"
df = exclude(df, m, "Test product (TEST001/TEST002)")
log("9 Exclude", "Test records", m.sum(), "Excluded")

# b. Negative prices: all are 'Adjust bad debt' (A-invoices, no customer)
m = df["UnitPrice"] < 0
df = exclude(df, m, "Negative unit price (bad-debt accounting adjustment)")
log("9 Exclude", "Negative UnitPrice", m.sum(), "Excluded")

m = df["InvoiceType"] == "Adjustment"
df = exclude(df, m, "'A' invoice - accounting adjustment, not a sale")
log("9 Exclude", "Remaining 'A' invoices (positive-price legs of bad-debt "
    "adjustments)", m.sum(), "Excluded")

# c. Zero price. 70%+ have no description, ~99% no customer and mostly
# negative qty on non-'C' invoices => warehouse stock write-offs
# ('damaged', 'check', 'thrown away', '?'), not commercial sales.
m = df["UnitPrice"] == 0
df = exclude(df, m, "Zero unit price (stock adjustment / write-off / free item)")
log("9 Exclude", "UnitPrice = 0", m.sum(), "Excluded")

# d. Negative quantity on a non-cancellation invoice (any survivors)
m = (df["Quantity"] < 0) & (df["InvoiceType"] != "Cancellation")
df = exclude(df, m, "Negative quantity on non-cancellation invoice")
log("9 Exclude", "Negative Quantity without 'C' prefix (after zero-price "
    "removal)", m.sum(), "Excluded")

# e. 'C' invoice with POSITIVE quantity (contradiction)
m = (df["InvoiceType"] == "Cancellation") & (df["Quantity"] > 0)
df = exclude(df, m, "Cancellation invoice with positive quantity")
log("9 Exclude", "'C' invoice with positive Quantity", m.sum(), "Excluded")


[9 Exclude] Test records: 17 rows -> Excluded
[9 Exclude] Negative UnitPrice: 5 rows -> Excluded
[9 Exclude] Remaining 'A' invoices (positive-price legs of bad-debt adjustments): 1 rows -> Excluded
[9 Exclude] UnitPrice = 0: 6,011 rows -> Excluded
[9 Exclude] Negative Quantity without 'C' prefix (after zero-price removal): 0 rows -> Excluded
[9 Exclude] 'C' invoice with positive Quantity: 1 rows -> Excluded


In [27]:
# DESCRIPTION CLEANING
# ----------------------------------------------------------------------------
# One stock code has many descriptions (typos, free-text notes such as
# 'WRONGLY MARKED CARTON 22804', '?', 'DAMAGED'). Build a master name per code
# = most frequent description among rows with a positive price.
valid = df[df["Description"].notna() & (df["UnitPrice"] > 0)]
n_variants = valid.groupby("StockCode")["Description"].nunique()
log("10 Description", "StockCodes with >1 description variant (after "
    "trimming/upper-casing)", (n_variants > 1).sum(),
    "Master name = most frequent description")

master = (valid.groupby(["StockCode", "Description"]).size()
          .reset_index(name="n")
          .sort_values(["StockCode", "n"], ascending=[True, False])
          .drop_duplicates("StockCode")
          .set_index("StockCode")["Description"])

orig = df["Description"].copy()
df["Description"] = df["StockCode"].map(master).fillna(df["Description"])
log("10 Description", "Description replaced by master name",
    ((orig != df["Description"]) & orig.notna()).sum(),
    "Standardised")
log("10 Description", "Missing Description filled from master (by StockCode)",
    (orig.isna() & df["Description"].notna()).sum(), "Imputed")

m = df["Description"].isna()
df = exclude(df, m, "Missing description and no master name for StockCode")
log("10 Description", "Still missing after fill", m.sum(), "Excluded")

[10 Description] StockCodes with >1 description variant (after trimming/upper-casing): 596 rows -> Master name = most frequent description
[10 Description] Description replaced by master name: 59,068 rows -> Standardised
[10 Description] Missing Description filled from master (by StockCode): 0 rows -> Imputed
[10 Description] Still missing after fill: 0 rows -> Excluded


In [28]:
# COUNTRY STANDARDISATION
# ----------------------------------------------------------------------------
df["CountryOriginal"] = df["Country"]
df["Country"] = df["Country"].replace(COUNTRY_MAP)
log("11 Country", "Non-standard names (EIRE, RSA, USA, Korea, Unspecified, "
    "European Community)", (df["Country"] != df["CountryOriginal"]).sum(),
    "Mapped to ISO-style names; 'Unspecified'/'European Community' -> 'Unknown'")

# Customers that appear under more than one country
ncountry = df.dropna(subset=["CustomerID"]).groupby("CustomerID")["Country"].nunique()
multi = ncountry[ncountry > 1].index
mode_country = (df.dropna(subset=["CustomerID"])
                .groupby("CustomerID")["Country"]
                .agg(lambda s: s.value_counts().index[0]))
df["CustomerCountry"] = df["CustomerID"].map(mode_country)
df["CustomerHasMultipleCountries"] = df["CustomerID"].isin(multi)
log("11 Country", "Customers linked to >1 country", len(multi),
    "Kept transaction Country; added CustomerCountry (most frequent) + flag")

[11 Country] Non-standard names (EIRE, RSA, USA, Korea, Unspecified, European Community): 19,241 rows -> Mapped to ISO-style names; 'Unspecified'/'European Community' -> 'Unknown'
[11 Country] Customers linked to >1 country: 13 rows -> Kept transaction Country; added CustomerCountry (most frequent) + flag


In [29]:
# DATE CHECKS & DERIVED TIME FIELDS
# ----------------------------------------------------------------------------
inv_span = df.groupby("InvoiceNo")["InvoiceDate"].nunique()
bad_inv = inv_span[inv_span > 1].index
min_ts = df.groupby("InvoiceNo")["InvoiceDate"].transform("min")
changed = (df["InvoiceDate"] != min_ts).sum()
df["InvoiceDate"] = min_ts
log("12 Date", "Invoices whose lines carry different timestamps (1-9 min "
    "apart)", len(bad_inv),
    f"Set every line to the invoice's first timestamp ({changed:,} lines changed)")

df["InvoiceDay"] = df["InvoiceDate"].dt.normalize()
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["Weekday"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour
log("12 Date", "Derived Year/Month/YearMonth/Weekday/Hour", len(df), "Added")
# Data observations (not errors):
#  - No Saturday trading except 5 Dec 2009 (402 rows) -> one-off.
#  - Dec 2011 is incomplete (data ends 9 Dec 2011) -> do not compare to Dec 2010.

[12 Date] Invoices whose lines carry different timestamps (1-9 min apart): 83 rows -> Set every line to the invoice's first timestamp (3,350 lines changed)
[12 Date] Derived Year/Month/YearMonth/Weekday/Hour: 1,026,997 rows -> Added


In [30]:
# MISSING CUSTOMER ID
# ----------------------------------------------------------------------------
# ~22% of lines (15% of revenue) have no CustomerID: guest / wholesale / web
# orders. Cannot be imputed. Keep for sales reporting; exclude from
# customer-level analysis (RFM, retention, CLV) via the flag.
df["HasCustomerID"] = df["CustomerID"].notna()
log("13 Customer", "Missing CustomerID", (~df["HasCustomerID"]).sum(),
    "NOT imputed. Flag HasCustomerID added; use it to filter customer analytics")

[13 Customer] Missing CustomerID: 229,200 rows -> NOT imputed. Flag HasCustomerID added; use it to filter customer analytics


In [31]:
# STEP 14 - MEASURES
# ----------------------------------------------------------------------------
df["LineTotal"] = (df["Quantity"] * df["UnitPrice"]).round(2)
df["IsReturn"] = df["Quantity"] < 0
log("14 Measures", "LineTotal = Quantity x UnitPrice (negative for cancellations)",
    len(df), "Added; IsReturn flag added")

[14 Measures] LineTotal = Quantity x UnitPrice (negative for cancellations): 1,026,997 rows -> Added; IsReturn flag added


In [32]:
# OUTLIER FLAGS (flag only, never delete)
# ----------------------------------------------------------------------------
prod = df["IsProduct"]
# Price: compare each line with the median price of the same StockCode.
med = df[prod].groupby("StockCode")["UnitPrice"].transform("median")
cnt = df[prod].groupby("StockCode")["UnitPrice"].transform("count")
ratio = df.loc[prod, "UnitPrice"] / med
df["PriceOutlier"] = False
df.loc[prod, "PriceOutlier"] = ((ratio > 10) | (ratio < 0.1)) & (cnt >= 5)
log("15 Outliers", "Unit price >10x or <0.1x the StockCode median "
    "(codes with >=5 lines)", df["PriceOutlier"].sum(), "Flagged PriceOutlier")

# Quantity: absolute quantity above the 99.9th percentile of product lines.
q_thr = df.loc[prod, "Quantity"].abs().quantile(0.999)
df["QuantityOutlier"] = prod & (df["Quantity"].abs() > q_thr)
log("15 Outliers", f"|Quantity| > P99.9 ({q_thr:,.0f})",
    df["QuantityOutlier"].sum(), "Flagged QuantityOutlier")

# Line value outliers (very large single lines).
v_thr = df.loc[prod, "LineTotal"].abs().quantile(0.999)
df["LineValueOutlier"] = prod & (df["LineTotal"].abs() > v_thr)
log("15 Outliers", f"|LineTotal| > P99.9 (GBP {v_thr:,.0f})",
    df["LineValueOutlier"].sum(), "Flagged LineValueOutlier")

# Same-day full reversals (e.g. 80,995 units sold and cancelled 12 min later)
sales = df[df["Quantity"] > 0][["CustomerID", "StockCode", "Quantity", "InvoiceDate"]]
canc = df[df["Quantity"] < 0].reset_index()
canc["AbsQty"] = -canc["Quantity"]
mm = canc.dropna(subset=["CustomerID"]).merge(
    sales.rename(columns={"Quantity": "SaleQty", "InvoiceDate": "SaleDate"}),
    on=["CustomerID", "StockCode"])
mm = mm[(mm["SaleQty"] == mm["AbsQty"]) & (mm["SaleDate"] <= mm["InvoiceDate"])
        & ((mm["InvoiceDate"] - mm["SaleDate"]) <= pd.Timedelta(days=1))]
df["IsSameDayReversal"] = df.index.isin(mm["index"].unique())
log("15 Outliers", "Cancellation fully reversing a sale within 24h "
    "(same customer, item, quantity)", df["IsSameDayReversal"].sum(),
    "Flagged IsSameDayReversal (consider excluding from volume KPIs)")

[15 Outliers] Unit price >10x or <0.1x the StockCode median (codes with >=5 lines): 915 rows -> Flagged PriceOutlier
[15 Outliers] |Quantity| > P99.9 (504): 995 rows -> Flagged QuantityOutlier
[15 Outliers] |LineTotal| > P99.9 (GBP 790): 1,022 rows -> Flagged LineValueOutlier
[15 Outliers] Cancellation fully reversing a sale within 24h (same customer, item, quantity): 1,448 rows -> Flagged IsSameDayReversal (consider excluding from volume KPIs)


In [33]:
# DIMENSION TABLES
# ----------------------------------------------------------------------------
dim_product = (df[df["UnitPrice"] > 0]
               .groupby("StockCode")
               .agg(Description=("Description", "first"),
                    LineCategory=("LineCategory", "first"),
                    MedianUnitPrice=("UnitPrice", "median"),
                    MinUnitPrice=("UnitPrice", "min"),
                    MaxUnitPrice=("UnitPrice", "max"),
                    FirstSold=("InvoiceDate", "min"),
                    LastSold=("InvoiceDate", "max"),
                    LineCount=("InvoiceNo", "size"))
               .reset_index())

cust = df.dropna(subset=["CustomerID"])
dim_customer = (cust.groupby("CustomerID")
                .agg(Country=("CustomerCountry", "first"),
                     FirstPurchase=("InvoiceDate", "min"),
                     LastPurchase=("InvoiceDate", "max"),
                     Invoices=("InvoiceNo", "nunique"),
                     NetRevenue=("LineTotal", "sum"))
                .reset_index())


In [34]:
# FINAL SHAPE, RECONCILIATION & EXPORT
# ----------------------------------------------------------------------------
excluded = pd.concat(excluded_parts, ignore_index=True)

final_cols = [
    "InvoiceNo", "InvoiceType", "InvoiceDate", "InvoiceDay", "Year", "Month",
    "YearMonth", "Weekday", "Hour", "StockCode", "Description", "LineCategory",
    "IsProduct", "Quantity", "UnitPrice", "LineTotal", "IsReturn",
    "CustomerID", "HasCustomerID", "Country", "CountryOriginal",
    "CustomerCountry", "CustomerHasMultipleCountries", "PriceOutlier",
    "QuantityOutlier", "LineValueOutlier", "IsSameDayReversal",
    "SourceSheet", "SourceRow",
]
fact = (df[final_cols].sort_values(["InvoiceDate", "InvoiceNo", "StockCode"])
        .reset_index(drop=True))

# Reconciliation: raw = clean + excluded
assert ROWS_RAW == len(fact) + len(excluded), "Row reconciliation failed"
log("17 Reconcile", f"Raw {ROWS_RAW:,} = clean {len(fact):,} + excluded "
    f"{len(excluded):,}", len(fact) + len(excluded), "Reconciled")



[17 Reconcile] Raw 1,067,371 = clean 1,026,997 + excluded 40,374: 1,067,371 rows -> Reconciled


In [39]:
# Final validation checks on the clean table
assert (fact["UnitPrice"] > 0).all()
assert fact["Description"].notna().all()
assert not fact.duplicated(dup_key).any()
assert ((fact["InvoiceType"] == "Cancellation") == (fact["Quantity"] < 0)).all()
print("All post-clean validation checks passed")

fact.to_csv(f"{OUT_DIR}/fact_transactions_clean.csv.gz", index=False,
            compression="gzip")
excluded[["InvoiceNo", "StockCode", "Description", "DescriptionRaw", "Quantity",
          "InvoiceDate", "UnitPrice", "CustomerID", "Country", "SourceSheet",
          "SourceRow", "ExclusionReason"]].to_csv(
    f"{OUT_DIR}/excluded_rows_audit.csv.gz", index=False, compression="gzip")
dim_product.to_csv(f"{OUT_DIR}/dim_product.csv", index=False)
dim_customer.to_csv(f"{OUT_DIR}/dim_customer.csv", index=False)
pd.DataFrame(issue_log).to_csv(f"{OUT_DIR}/issue_log.csv", index=False)

print("\nSummary of exclusions:")
print(excluded["ExclusionReason"].value_counts().to_string())


All post-clean validation checks passed

Summary of exclusions:
Cross-tab duplicate (1-9 Dec 2010 present in both tabs)                    22523
Duplicate line (same invoice, item, quantity, price, customer, country)    11816
Zero unit price (stock adjustment / write-off / free item)                  6011
Test product (TEST001/TEST002)                                                17
Negative unit price (bad-debt accounting adjustment)                           5
'A' invoice - accounting adjustment, not a sale                                1
Cancellation invoice with positive quantity                                    1
